In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [2]:
customers_df = pd.read_csv("Customers1.csv")

accounts_df = pd.read_csv("Accounts.csv")

In [3]:
customers_df["DOB"] = pd.to_datetime(customers_df["DOB"])

today = pd.Timestamp.today()

customers_df["Age"] = (
    (today - customers_df["DOB"]).dt.days // 365
)

In [4]:
active_customers = accounts_df[
    accounts_df["Status"] == "Active"
]["Customer_ID"].unique()

customers_df = customers_df[
    customers_df["Customer_ID"].isin(active_customers)
].copy()

In [5]:
eligible_loan_customers = customers_df[

    (customers_df["Age"] >= 21) &
    (customers_df["Age"] <= 70) &
    (customers_df["Annual_Income"] >= 250000)

].copy()

In [6]:
eligible_loan_customers.shape

(8235, 15)

In [7]:
loan_customers = eligible_loan_customers.sample(
    n=2500,
    random_state=42
).reset_index(drop=True)

In [8]:
loan_customers.shape

(2500, 15)

In [9]:
LOAN_RULES = {

    "Personal": {

        "amount": (100000, 1500000),

        "interest": (10.5, 18.0)

    },

    "Home": {

        "amount": (1500000, 10000000),

        "interest": (7.5, 10.5)

    },

    "Auto": {

        "amount": (300000, 2500000),

        "interest": (8.5, 12.5)

    },

    "Education": {

        "amount": (200000, 3000000),

        "interest": (8.0, 11.0)

    }

}

In [10]:
LOAN_STATUS = [

    "Active",
    "Closed"

]

LOAN_STATUS_WEIGHTS = [

    75,
    25

]

In [11]:
LOAN_TYPES = [

    "Personal",
    "Home",
    "Auto",
    "Education"

]

LOAN_WEIGHTS = [

    40,
    25,
    20,
    15

]

In [12]:
loan_customers.shape

(2500, 15)

In [13]:
loan_customers.head()

,Customer_ID,First_Name,Last_Name,Gender,DOB,Email,Phone,City,State,Occupation,Annual_Income,Marital_Status,Customer_Segment,Join_Date,Age
0,C00876,Zayyan,Pandya,Female,1996-02-13,zayyan.pandya165@email.com,9057210014,Mumbai,Maharashtra,Doctor,1956147,Married,VIP,2020-11-11,30
1,C07327,Aadi,Devan,Male,1984-05-21,aadi.devan989@email.com,8354553089,Chennai,Tamil Nadu,Teacher,388511,Married,Retail,2018-09-29,42
2,C09308,Harinakshi,Krish,Male,1967-06-20,harinakshi.krish807@email.com,8204766314,Delhi,Delhi,Accountant,434237,Married,Retail,2019-11-25,59
3,C08461,Omya,Bhakta,Male,1971-10-12,omya.bhakta2@email.com,9831335596,Ahmedabad,Gujarat,Software Engineer,816905,Single,Premium,2022-08-15,54
4,C06821,Samuel,Kibe,Male,2001-06-24,samuel.kibe849@email.com,9632867130,Hyderabad,Telangana,Doctor,1925274,Divorced,VIP,2024-03-29,25


In [14]:
loans = []

loan_number = 1

In [15]:
for _, customer in loan_customers.iterrows():

    customer_id = customer["Customer_ID"]
    age = customer["Age"]
    income = customer["Annual_Income"]
    join_date = pd.to_datetime(customer["Join_Date"])

    # ------------------------
    # Choose Loan Type
    # ------------------------

    if age <= 30:
        loan_type = random.choices(
            ["Education", "Personal", "Auto"],
            weights=[40, 40, 20],
            k=1
        )[0]

    elif age <= 45:
        loan_type = random.choices(
            ["Personal", "Auto", "Home"],
            weights=[45, 25, 30],
            k=1
        )[0]

    else:
        loan_type = random.choices(
            ["Home", "Personal", "Auto"],
            weights=[45, 35, 20],
            k=1
        )[0]

    rules = LOAN_RULES[loan_type]

    # ------------------------
    # Loan Amount
    # ------------------------

    max_by_income = income * 5

    lower = rules["amount"][0]

    upper = min(rules["amount"][1], max_by_income)

    if upper < lower:
        upper = lower

    loan_amount = random.randint(
        int(lower),
        int(upper)
    )

    # ------------------------
    # Interest Rate
    # ------------------------

    interest_rate = round(
        random.uniform(
            rules["interest"][0],
            rules["interest"][1]
        ),
        2
    )

    # ------------------------
    # Loan Duration
    # ------------------------

    tenure_years = random.choice(
        [5, 10, 15, 20]
    )

    months = tenure_years * 12

    # ------------------------
    # EMI Calculation
    # ------------------------

    r = interest_rate / (12 * 100)

    emi = (
        loan_amount * r *
        ((1 + r) ** months)
    ) / (
        ((1 + r) ** months) - 1
    )

    emi = round(emi, 2)

    # ------------------------
    # Loan Dates
    # ------------------------

    start_date = fake.date_between(
        start_date=join_date.date(),
        end_date="today"
    )

    end_date = (
        pd.to_datetime(start_date) +
        pd.DateOffset(years=tenure_years)
    )

    # ------------------------
    # Loan Status
    # ------------------------

    loan_status = random.choices(
        LOAN_STATUS,
        weights=LOAN_STATUS_WEIGHTS,
        k=1
    )[0]

    # ------------------------
    # Remaining Balance
    # ------------------------

    if loan_status == "Closed":
        remaining_balance = 0

    else:
        paid_fraction = random.uniform(
            0.10,
            0.80
        )

        remaining_balance = round(
            loan_amount * (1 - paid_fraction),
            2
        )

    # ------------------------
    # Loan ID
    # ------------------------

    loan_id = f"L{loan_number:06d}"

    loan_number += 1

    loans.append({

        "Loan_ID": loan_id,

        "Customer_ID": customer_id,

        "Loan_Type": loan_type,

        "Loan_Amount": loan_amount,

        "Interest_Rate": interest_rate,

        "EMI": emi,

        "Remaining_Balance": remaining_balance,

        "Loan_Status": loan_status,

        "Start_Date": start_date,

        "End_Date": end_date.date()

    })

In [16]:
loans_df = pd.DataFrame(loans)

loans_df.shape

(2500, 10)

In [17]:
loans_df.head()

,Loan_ID,Customer_ID,Loan_Type,Loan_Amount,Interest_Rate,EMI,Remaining_Balance,Loan_Status,Start_Date,End_Date
0,L000001,C00876,Personal,152451,16.06,2559.46,58612.86,Active,2026-04-20,2036-04-20
1,L000002,C07327,Auto,1443716,8.85,12850.54,1204655.88,Active,2019-09-09,2039-09-09
2,L000003,C09308,Home,2131262,7.58,25387.53,871840.71,Active,2020-02-10,2030-02-10
3,L000004,C08461,Home,3384119,9.27,70693.03,0.00,Closed,2025-10-09,2030-10-09
4,L000005,C06821,Education,1972573,9.02,25009.08,721082.17,Active,2025-05-30,2035-05-30


In [18]:
loans_df["Loan_ID"].duplicated().sum()

np.int64(0)

In [19]:
loans_df.isnull().sum()

Loan_ID              0
Customer_ID          0
Loan_Type            0
Loan_Amount          0
Interest_Rate        0
EMI                  0
Remaining_Balance    0
Loan_Status          0
Start_Date           0
End_Date             0
dtype: int64

In [20]:
loans_df["Loan_Type"].value_counts()

Loan_Type
Personal     960
Home         791
Auto         561
Education    188
Name: count, dtype: int64

In [21]:
loans_df["Loan_Status"].value_counts(normalize=True) * 100

Loan_Status
Active    76.4
Closed    23.6
Name: proportion, dtype: float64

In [22]:
(loans_df["Remaining_Balance"] <= loans_df["Loan_Amount"]).all()

np.True_

In [23]:
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            2500 non-null   object 
 1   Customer_ID        2500 non-null   object 
 2   Loan_Type          2500 non-null   object 
 3   Loan_Amount        2500 non-null   int64  
 4   Interest_Rate      2500 non-null   float64
 5   EMI                2500 non-null   float64
 6   Remaining_Balance  2500 non-null   float64
 7   Loan_Status        2500 non-null   object 
 8   Start_Date         2500 non-null   object 
 9   End_Date           2500 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 195.4+ KB


In [24]:
loans_df["Start_Date"] = pd.to_datetime(loans_df["Start_Date"])
loans_df["End_Date"] = pd.to_datetime(loans_df["End_Date"])

In [25]:
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Loan_ID            2500 non-null   object        
 1   Customer_ID        2500 non-null   object        
 2   Loan_Type          2500 non-null   object        
 3   Loan_Amount        2500 non-null   int64         
 4   Interest_Rate      2500 non-null   float64       
 5   EMI                2500 non-null   float64       
 6   Remaining_Balance  2500 non-null   float64       
 7   Loan_Status        2500 non-null   object        
 8   Start_Date         2500 non-null   datetime64[ns]
 9   End_Date           2500 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(3), int64(1), object(4)
memory usage: 195.4+ KB


In [26]:
loans_df.to_csv(
    "Loans.csv",
    index=False
)